# Evidence Observer Notebook

This notebook provides a lightweight **evidence observer** to validate test runs.

It helps you:
- Run one or more test commands (for example: `pytest`, `npm test`, custom scripts)
- Capture stdout, stderr, exit code, and elapsed time
- Save an evidence report as JSON for auditability
- Add simple assertions to unit test the observed results


In [ ]:
from __future__ import annotations

import json
import subprocess
import time
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Iterable


In [ ]:
@dataclass
class EvidenceRecord:
    command: str
    return_code: int
    elapsed_seconds: float
    stdout: str
    stderr: str
    observed_at_utc: str


def run_command_with_evidence(command: str, timeout_seconds: int = 300) -> EvidenceRecord:
    start = time.perf_counter()
    proc = subprocess.run(
        command,
        shell=True,
        capture_output=True,
        text=True,
        timeout=timeout_seconds,
    )
    elapsed = time.perf_counter() - start

    return EvidenceRecord(
        command=command,
        return_code=proc.returncode,
        elapsed_seconds=round(elapsed, 4),
        stdout=proc.stdout,
        stderr=proc.stderr,
        observed_at_utc=datetime.now(timezone.utc).isoformat(),
    )


def observe_test_suite(commands: Iterable[str], timeout_seconds: int = 300) -> list[EvidenceRecord]:
    records: list[EvidenceRecord] = []
    for command in commands:
        print(f"Running: {command}")
        record = run_command_with_evidence(command, timeout_seconds=timeout_seconds)
        records.append(record)
        print(f" -> exit={record.return_code}, elapsed={record.elapsed_seconds}s")
    return records


def save_evidence(records: list[EvidenceRecord], output_path: str = "test-evidence.json") -> Path:
    out = Path(output_path)
    payload = {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "records": [asdict(r) for r in records],
    }
    out.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(f"Evidence written to: {out.resolve()}")
    return out


In [ ]:
# Configure test commands for this repository.
# Replace these with your actual test commands when you add tests.
TEST_COMMANDS = [
    "python -m unittest discover -v",
]

records = observe_test_suite(TEST_COMMANDS, timeout_seconds=300)
evidence_file = save_evidence(records, output_path="test-evidence.json")
records


In [ ]:
# Basic unit-test style assertions over observed evidence.
# Set this to True once the repository contains real tests.
REQUIRE_AT_LEAST_ONE_TEST = False

for record in records:
    no_tests_ran = (
        "NO TESTS RAN" in record.stdout
        or "Ran 0 tests" in record.stdout
    )

    if REQUIRE_AT_LEAST_ONE_TEST:
        assert not no_tests_ran, (
            f"No tests executed for command: {record.command}\n"
            f"STDOUT:\n{record.stdout}"
        )

    assert record.return_code in (0, 5), (
        f"Unexpected failure: {record.command}\n"
        f"STDOUT:\n{record.stdout}\n"
        f"STDERR:\n{record.stderr}"
    )

print("Evidence checks completed. Set REQUIRE_AT_LEAST_ONE_TEST=True when tests are added.")
